In [1]:
import pandas as pd

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


The class Ride is a blueprint or a template, not data itself. It says: "Any Ride object must have exactly these 5 named slots, and here's what type of value belongs in each slot." No actual ride exists yet — you've just described the shape a ride object will have

In [2]:
from dataclasses import dataclass

@dataclass
class Ride:
    PULocationID: int
    DOLocationID: int
    trip_distance: float
    total_amount: float
    tpep_pickup_datetime: int  # epoch milliseconds

You can pull individual values out of it with bracket access: row['PULocationID'] gives you 132, row['trip_distance'] gives you 2.5, and so on — this works because pandas rows let you access values by column name, similar to a dictionary

ride_from_row takes the row, pull each named value out of it, cast it to the right type, and use those 5 values to construct a Ride object.

In [3]:
def ride_from_row(row):
    return Ride(
        PULocationID=int(row['PULocationID']),
        DOLocationID=int(row['DOLocationID']),
        trip_distance=float(row['trip_distance']),
        total_amount=float(row['total_amount']),
        tpep_pickup_datetime=int(row['tpep_pickup_datetime'].timestamp() * 1000),
    )

In [4]:
ride = ride_from_row(df.iloc[0])
ride

Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)

In [13]:
ride.PULocationID

43

'**json_serializer**' is a function, we're just describing what should happen whenever this function gets called later, and with what input. Note the parameter name **data** — deliberately generic, because at this point the function doesn't know or care whether it'll receive a Ride, a **dict**, or **anything else**. It just assumes: "whatever I'm given, I can call json.dumps() on it."

json.dumps(data) converts a Python value into a **JSON-formatted string**. .encode('utf-8') then converts that string into **bytes**.

**Why bytes?**

Kafka write the information as bytes on disk. Kafka messages are just bytes — usually JSON strings under the hood

In [14]:
# serializer
import json

def json_serializer(data):
    return json.dumps(data).encode('utf-8')

This builds a **producer** object connected to the broker '**REDPANDA**' at localhost:**9092**, and tells it: "whenever I ask you to send something, run it through **json_serializer** first."  

In [15]:
from kafka import KafkaProducer
server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=json_serializer
)

**dataclasses.asdict(ride)** This converts our Ride object into a plain dict:
{
    'PULocationID': 132,
    'DOLocationID': 239,
    'trip_distance': 2.5,
    'total_amount': 14.5,
    'tpep_pickup_datetime': 1762028132000
}

**producer.send(topic_name, value=<that dict>)** The producer takes the dict we just built and passes it into **json_serializer** Inside this function it is converted to a **json.dumps(data) → the string**

'{"PULocationID": 132, "DOLocationID": 239, "trip_distance": 2.5, "total_amount": 14.5, "tpep_pickup_datetime": 1762028132000}'

**.encode('utf-8')** This encode the string into bytes

    b'{"PULocationID": 132, "DOLocationID": 239, "trip_distance": 2.5, "total_amount": 14.5, "tpep_pickup_datetime": 1762028132000}'
    
Those bytes are what actually get transmitted to the broker on topic 'rides', queued internally by the producer. **producer.flush(**) then blocks until the broker has confirmed it actually received and stored that message

In [16]:
import dataclasses

topic_name = 'rides'

producer.send(topic_name, value=dataclasses.asdict(ride))
producer.flush()

**dataclasses.asdict(ride)** appears: outside json_serializer, sitting right there in the producer.send(...) call itself. json_serializer never touches Ride objects or asdict() — it only knows how to handle things that are already **dict-shaped**.

This means: every single time anywhere in the code you want to send a Ride, **you must remember to write dataclasses.asdict(ride)** yourself before handing it to producer.send(). If you forget — say, you write producer.send(topic_name, value=ride) (passing the raw Ride object) — json_serializer would receive the actual Ride object as data, call json.dumps(ride) on it, and that would crash with a **TypeError**, because json.dumps() has no idea how to serialize a custom Ride object; it only knows built-in types like dicts, lists, strings, and numbers.

In [17]:
# new function which converts the object-->to dict--> then to string --> bytes
# serializer -version 2
def ride_serializer(ride):
    ride_dict = dataclasses.asdict(ride)
    json_str = json.dumps(ride_dict)
    return json_str.encode('utf-8')

In [18]:
# recreate the producer with the new serializer. 
# we can pass Ride objects directly without converting them to dicts first
producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [19]:
#Send one ride to verify
producer.send(topic_name, value=ride)
producer.flush()

# producer.flush()
producer.send() does not open a network connection and transmit those bytes right then and there. Instead, it does something more like this internally:

Runs your value through the serializer → gets bytes (as we traced above).
1) Drops those bytes into an internal memory buffer (sometimes called a "record accumulator"),
2) grouped by which partition they're headed to.
3) Returns immediately — handing you back a Future object (a placeholder for "the result of this send, once it actually happens").

**KafkaProducer** runs a background thread that periodically wakes up and does the real network work: it takes whatever's sitting in that buffer, batches multiple messages together (for efficiency — one network trip carrying many messages is much cheaper than one trip per message), and sends the batch to the broker over the network connection.

When **producer.send()** sending 1000 messages with time.sleep(0.01) between each doesn't mean 1000 separate network round-trips happening synchronously in your loop. Your loop is just filling a buffer; the actual network transmission is happening concurrently, on its own schedule, in the background.

If there is no flush(), then after send() the script ends immediately here. Send() would have copied the bytes into the buffer but background thread would have not copied into the broker over the network and the process end and the buffer is deleted. The broker thinks evethging that has been processed has been sent because it doesn#t know about the missing bytes.

**producer.flush()**
This call blocks — your script pauses right here and does nothing else — until every message currently sitting in the internal buffer has:

1) Actually been transmitted over the network to the broker, and
2) Been acknowledged by the broker (the broker has confirmed "yes, I received this and wrote it to my log").

producer.send(topic_name, value=ride), our one Ride for 'PULocationID 132' has been serialized to bytes and dropped into the buffer. Then producer.flush() sits and waits — actively forcing the background thread to send that buffered batch right now rather than waiting for its normal schedule — until the broker sends back confirmation. Only after that confirmation arrives does flush() return and let your script continue (or exit).




In [21]:
#  Now let's send all 1000 rides in a loop
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    #print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 10.79 seconds
